# Linear Regression – Ecommerce Customers
**Task:** Predict the **Yearly Amount Spent** by a customer based on their behaviour metrics.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('Ecommerce Customers')
print('Dataset shape:', df.shape)

## 3. Explore the Data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**Observations:**
- The dataset has **500 rows** and **8 columns**.
- Columns `Email`, `Address`, and `Avatar` are non-numeric (categorical/text).
- The four numeric feature columns (`Avg. Session Length`, `Time on App`, `Time on Website`, `Length of Membership`) and the target `Yearly Amount Spent` are all `float64`.
- All values appear to be in a similar scale (~12–40 range for features, ~250–765 for target).

## 4. Basic Data Cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

**Result:** No missing values — the dataset is clean and requires no imputation or row removal.

In [ ]:
# Check for duplicate rows
print('Duplicate rows:', df.duplicated().sum())

## 5. Feature Engineering

The columns `Email`, `Address`, and `Avatar` are identifiers/categorical with high cardinality and no direct linear relationship with spending. We **drop** them and keep only the four numeric behavioural features as predictors.

In [ ]:
# Visualise correlations among numeric columns
corr = df[['Avg. Session Length', 'Time on App', 'Time on Website',
           'Length of Membership', 'Yearly Amount Spent']].corr()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

**Key insight:** `Length of Membership` has the strongest correlation with `Yearly Amount Spent` (~0.81), followed by `Time on App` (~0.50). `Time on Website` has almost no linear correlation (~0.01), which will be reflected in its model coefficient.

## 6. Prepare the Data for Modelling

In [ ]:
# Define features (X) and target (y)
features = ['Avg. Session Length', 'Time on App', 'Time on Website', 'Length of Membership']

X = df[features]
y = df['Yearly Amount Spent']

# Split: 70% training, 30% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print('Training samples :', X_train.shape[0])
print('Testing  samples :', X_test.shape[0])

## 7. Train the Model

In [ ]:
# Instantiate and fit a Multiple Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print('Intercept: {:.4f}'.format(model.intercept_))
print('\nCoefficients:')
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_})
print(coef_df.to_string(index=False))

**Interpretation of coefficients:**
| Feature | Coefficient | Meaning |
|---|---|---|
| Avg. Session Length | 25.72 | Each extra minute of session → +$25.72 spent |
| Time on App | 38.60 | Each extra minute on app → +$38.60 spent |
| Time on Website | 0.46 | Very small effect; almost no linear relationship |
| Length of Membership | 61.67 | Each extra year of membership → +$61.67 spent |

## 8. Evaluate Model Performance

In [ ]:
# Generate predictions on the test set
y_pred = model.predict(X_test)

# Compute metrics
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=== Model Performance (Test Set) ===')
print(f'  MAE  : {mae:.4f}')
print(f'  MSE  : {mse:.4f}')
print(f'  RMSE : {rmse:.4f}')
print(f'  R²   : {r2:.4f}')

**Results summary:**
| Metric | Value | Interpretation |
|---|---|---|
| MAE | 8.43 | On average, predictions are off by ~$8.43 |
| RMSE | 10.19 | Typical prediction error is ~$10.19 |
| R² | 0.9809 | The model explains **98.09%** of the variance in spending |

In [ ]:
# Plot: Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.5, edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', linewidth=1.5, label='Perfect fit')
plt.xlabel('Actual Yearly Amount Spent ($)')
plt.ylabel('Predicted Yearly Amount Spent ($)')
plt.title('Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Residuals distribution
residuals = y_test - y_pred

plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=30, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--')
plt.xlabel('Residual ($)')
plt.title('Distribution of Residuals')
plt.tight_layout()
plt.show()

print('Residual mean  :', round(residuals.mean(), 4))
print('Residual std   :', round(residuals.std(),  4))

## 9. Conclusion

The **Multiple Linear Regression** model performs extremely well on this dataset:

- An **R² of ~0.98** indicates the four behavioural features capture almost all of the variation in annual customer spending.
- The average prediction error (RMSE ≈ \$10) is small relative to the target range (~\$250–\$765).
- Residuals are approximately normally distributed and centred near zero, confirming the linear model assumptions are reasonably satisfied.
- **`Length of Membership`** is the most influential predictor, followed by **`Time on App`**, suggesting the company should focus on retaining customers and improving the mobile app experience to increase revenue.